# 02 -- Shala Score v2

**Purpose:** Explore the 5-pillar weighted Shala Score v2, its distribution,
risk tiers, and PCA diagnostic.

**Prerequisites:** Run `src/features/build_shala_score_v2.py` first.

**Key file:** `data/processed/district_year/maharashtra_district_year_shala_score_v2.csv`

**Pillar design (5 pillars, Learning = MISSING_IN_SOURCE):**

| Pillar | Nominal weight | Effective weight | Status |
|---|---|---|---|
| Teacher PTR | 25% | 29.4% | AVAILABLE |
| Infrastructure | 25% | 29.4% | AVAILABLE |
| Participation | 25% | 23.5% | AVAILABLE |
| Digital | 25% | 17.6% | AVAILABLE |
| Learning | 15% | 0% | **MISSING_IN_SOURCE** |

No double-counting: `facility_classrooms_good_condition_pct` is in Infrastructure only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
V2   = ROOT / 'data/processed/district_year/maharashtra_district_year_shala_score_v2.csv'
df   = pd.read_csv(V2, low_memory=False)

score_min = df['shala_score_v2'].min()
score_max = df['shala_score_v2'].max()
print(f'Shape: {df.shape}')
print(f'Score range: {score_min:.2f} - {score_max:.2f}  (expected approx 24.90 - 72.16)')

In [ ]:
# Score distribution
df['shala_score_v2'].hist(bins=20, color='#0F766E', edgecolor='white')
plt.xlabel('Shala Score v2')
plt.title('Distribution of Shala Score v2 (288 district-years)')
plt.tight_layout()
plt.show()

In [ ]:
# Risk tier counts -- must be 72 each (equal-size quartile bands)
tier_counts = df['risk_tier_v2'].value_counts()
print('Risk tier distribution (expected: 72 each):')
print(tier_counts.to_string())
assert (tier_counts == 72).all(), 'ERROR: tiers are not equal-size -- check build_shala_score_v2.py'

In [ ]:
# Pillar correlation matrix
pillar_cols = ['pillar2_teacher_ptr', 'pillar2_infrastructure',
               'pillar2_participation', 'pillar2_digital']
print('Pillar correlation matrix:')
print(df[pillar_cols].corr().round(3))

In [ ]:
# No double-counting check: pillar2_learning must be absent or all-NaN
if 'pillar2_learning' in df.columns:
    n_non_null = df['pillar2_learning'].notna().sum()
    if n_non_null > 0:
        print(f'WARNING: pillar2_learning has {n_non_null} non-null values -- check for double-counting')
    else:
        print('OK: pillar2_learning is present but all-NaN (MISSING_IN_SOURCE, correctly handled)')
else:
    print('OK: pillar2_learning column absent (MISSING_IN_SOURCE)')

# No synthetic data check
FORBIDDEN = ['learning_outcome', 'ger', 'dropout_rate', 'aser_score', 'nas_score']
for col in FORBIDDEN:
    if col in df.columns and pd.to_numeric(df[col], errors='coerce').notna().any():
        print(f'WARNING: {col} has non-null values -- this column should be MISSING_IN_SOURCE')
    else:
        print(f'OK: {col} absent or all-NaN (MISSING_IN_SOURCE confirmed)')

In [ ]:
# Top 10 districts in the most recent year
latest = df['academic_year'].max()
top10  = (
    df[df['academic_year'] == latest]
    .nlargest(10, 'shala_score_v2')
    [['district', 'shala_score_v2', 'risk_tier_v2']]
)
print(f'Top 10 districts in {latest}:')
print(top10.to_string(index=False))

In [ ]:
# Score trend over all years
trend = df.groupby('academic_year')['shala_score_v2'].mean()
trend.sort_index().plot(marker='o', color='#0F766E', title='Mean Shala Score v2 by Year')
plt.ylabel('Shala Score v2')
plt.xlabel('Academic Year')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()